# Phase 3A — Exploratory Data Analysis

This notebook profiles `data/processed/features.csv` before any model training. We answer four questions:

1. **What does the dataset look like?** Shape, dtypes, target distribution.
2. **What does the group structure look like?** Queries × variants.
3. **Which features carry signal?** Correlation with the target, split into plan-time vs leaky.
4. **What does the variant axis tell us?** When PG is forced off a join strategy, does the runtime actually move?

Run each cell top-to-bottom. Static PNG copies of every plot are also written to `reports/phase3a/plots/` by `phase3a/reports.py`.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# make `from phase3a... import ...` work when running from notebooks/
ROOT = Path.cwd()
while not (ROOT / 'phase3a').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from phase3a.feature_selection import (
    ID_COLUMNS, LEAKY_COLUMNS, TARGET_COLUMN, GROUP_COLUMN,
    describe_regime_split, build_feature_matrix,
)

sns.set_theme(context='notebook', style='whitegrid')
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 160)

FEATURES_CSV = ROOT / 'data' / 'processed' / 'features.csv'
df = pd.read_csv(FEATURES_CSV)
print(f'rows: {len(df):,}   cols: {df.shape[1]}')
df.head()

## 1. Dataset profile

How many plans, how many queries, how many variants?

In [ ]:
print(f'distinct query_ids : {df[GROUP_COLUMN].nunique()}')
print(f'distinct variants  : {df.variant.nunique()}')
print(f'distinct tags      : {df.tag.nunique()}')
print()
print('plans per query_id (head):')
print(df.groupby(GROUP_COLUMN).size().describe())
print()
print('plans per variant:')
print(df.variant.value_counts())

In [ ]:
y = df[TARGET_COLUMN].astype(float)
summary = pd.Series({
    'count':  len(y),
    'min':    y.min(),
    'p25':    y.quantile(0.25),
    'median': y.median(),
    'mean':   y.mean(),
    'p75':    y.quantile(0.75),
    'p95':    y.quantile(0.95),
    'max':    y.max(),
    'orders_of_magnitude': float(np.log10(max(y.max(), 1.0)) - np.log10(max(y.min(), 0.1))),
})
print('execution_time_ms:')
print(summary.round(2))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(y, bins=30, kde=True, ax=axes[0], color='steelblue')
axes[0].set_title('execution_time_ms — linear')
axes[0].set_xlabel('execution_time_ms')

sns.histplot(np.log10(y.clip(lower=0.1)), bins=30, kde=True, ax=axes[1], color='indianred')
axes[1].set_title('execution_time_ms — log10')
axes[1].set_xlabel('log10(execution_time_ms)')
fig.tight_layout()

Several orders of magnitude separating fast and slow queries → we will train models on `log1p(target)`.

## 2. Feature regime split

Each column lands in one of four buckets. The training pipeline drops `identifier` and `target`, drops `leaky` for the realistic regime, and uses everything else as `X`.

In [ ]:
regime_split = describe_regime_split(df)
regime_split.groupby('category').size()

In [ ]:
regime_split[regime_split.category != 'plan-time feature']

## 3. Correlation with target

Leaky columns are highlighted in red — they correlate strongly because they *are* the target. The plan-time features are the ones a real cost model has to learn from.

In [ ]:
drop = list(ID_COLUMNS)
work = df.drop(columns=drop, errors='ignore').apply(pd.to_numeric, errors='coerce')
work = work.dropna(axis=1, how='all')

corr_with_target = (
    work.corr(numeric_only=True)[TARGET_COLUMN]
        .drop(labels=[TARGET_COLUMN], errors='ignore')
        .abs()
        .sort_values(ascending=False)
        .head(20)
)
is_leaky = corr_with_target.index.to_series().isin(LEAKY_COLUMNS)
ax = corr_with_target.plot.barh(
    figsize=(8, 7),
    color=is_leaky.map({True: 'crimson', False: 'steelblue'}).to_numpy(),
)
ax.invert_yaxis()
ax.set_title('Top-20 |correlation| with execution_time_ms\n(red = leaky / post-execution)')
ax.set_xlabel('|Pearson correlation|')
plt.tight_layout()

In [ ]:
top = corr_with_target.head(20).index.tolist()
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(
    work[top + [TARGET_COLUMN]].corr(numeric_only=True),
    annot=True, fmt='.2f', cmap='coolwarm', center=0,
    square=True, cbar_kws={'shrink': 0.7}, ax=ax,
)
ax.set_title('Correlation heatmap — top-20 features + target')
fig.tight_layout()

## 4. Variant axis — do the optimizer knobs actually move runtime?

For each query we collected up to four variants (`default`, `no_hashjoin`, `no_mergejoin`, `no_nestloop`). If a variant collapses to the same plan as default, its runtime should be similar; if it forces a different plan, runtime should diverge.

We compute the per-query coefficient of variation of `execution_time_ms` across variants — a high number means the optimizer knob *did* something measurable.

In [ ]:
variant_stats = (
    df.groupby(GROUP_COLUMN)[TARGET_COLUMN]
      .agg(['count', 'mean', 'std', 'min', 'max'])
)
variant_stats['cv'] = variant_stats['std'] / variant_stats['mean']
variant_stats['range_ratio'] = variant_stats['max'] / variant_stats['min'].replace(0, np.nan)
variant_stats = variant_stats.sort_values('range_ratio', ascending=False)
variant_stats.head(10).round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
(variant_stats['range_ratio']
    .sort_values()
    .plot.bar(ax=ax, color='teal'))
ax.axhline(1.0, color='black', lw=0.8, ls='--')
ax.set_yscale('log')
ax.set_ylabel('max / min execution_time_ms\n(log scale)')
ax.set_xlabel('query_id')
ax.set_title('Per-query runtime spread across optimizer variants')
plt.tight_layout()

## 5. Plan-shape diversity per query

A query whose four variants all picked the same `root_node_type` is essentially a single sample for the model, no matter how many rows the CSV has.

In [ ]:
shape_diversity = (
    df.groupby(GROUP_COLUMN)
      .agg(distinct_root_node_types=('root_node_type', 'nunique'),
           total_plans=('root_node_type', 'size'))
      .sort_values(['distinct_root_node_types', 'total_plans'], ascending=[False, False])
)
shape_diversity

## 6. Sanity check: the realistic feature matrix

In [ ]:
fm = build_feature_matrix(df, regime='plan_time')
print(f'X shape : {fm.X.shape}')
print(f'y len   : {len(fm.y)}')
print(f'groups  : {fm.groups.nunique()} unique queries')
print()
print('first 8 features:')
print(fm.feature_names[:8])

All clear — ready to run `python phase3a/train_models.py` for the actual fit + cross-validation.